# A7 · Finding molecular families: K-means, random forests, and what 8 test molecules can say
**Series A: Machine Learning Basics for Chemists, Post 7 of 7** &nbsp;|&nbsp; companion *primer* notebook

Section 3.4 of the notes gives a tutorial and assignment, *Discovering Molecular Families with Unsupervised Learning*: 40 small molecules (hydrocarbons, alcohols, carboxylic acids, aromatics) are described by eight RDKit descriptors, standardized, viewed with PCA, grouped by K-means with $K$ chosen on a validation set, and compared with a supervised random forest and with a random-forest *proximity* construction that needs no labels. This primer rebuilds the **whole chain with NumPy only**: the descriptors (a SMILES parser and Crippen's logP), scikit-learn's stratified split, standardization, PCA, K-means, silhouette, ARI, and a random forest written from decision trees. It **reproduces every number the notebook printed** (PCA variances and loadings, the inertia and silhouette of every $K$, the cluster means, the ARI $0.588$, the accuracy $0.75$), and then asks what a test set of eight molecules can and cannot support. Part II checks each function against scikit-learn and RDKit.

> **Two parts.** **Part I** needs only NumPy, pandas and matplotlib and runs in well under a minute on a CPU; the 40 molecules are embedded. **Part II** needs scikit-learn and RDKit (installed by the cell).
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | descriptors from SMILES, without RDKit |
| 2 | the stratified 60/20/20 split |
| 3 | standardization and PCA |
| 4 | K-means, silhouette, and the choice of $K$ |
| 5 | ARI, the post-hoc mapping, and what the families look like |
| 6 | random forests: supervised, and the proximity construction |
| 7 | how much can 8 test molecules say? |
| **Part II** | cross-check with scikit-learn and RDKit |

## 0. Setup and data

In [ ]:
import math, time, importlib.util
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
SEED = 42

In [ ]:
records = [
("methane", "C", "hydrocarbon"),
("ethane", "CC", "hydrocarbon"),
("propane", "CCC", "hydrocarbon"),
("butane", "CCCC", "hydrocarbon"),
("pentane", "CCCCC", "hydrocarbon"),
("hexane", "CCCCCC", "hydrocarbon"),
("heptane", "CCCCCCC", "hydrocarbon"),
("octane", "CCCCCCCC", "hydrocarbon"),
("cyclopentane", "C1CCCC1", "hydrocarbon"),
("cyclohexane", "C1CCCCC1", "hydrocarbon"),
("methanol", "CO", "alcohol"),
("ethanol", "CCO", "alcohol"),
("1-propanol", "CCCO", "alcohol"),
("1-butanol", "CCCCO", "alcohol"),
("1-pentanol", "CCCCCO", "alcohol"),
("1-hexanol", "CCCCCCO", "alcohol"),
("2-propanol", "CC(C)O", "alcohol"),
("2-butanol", "CCC(C)O", "alcohol"),
("ethylene glycol", "OCCO", "alcohol"),
("glycerol", "OCC(O)CO", "alcohol"),
("formic acid", "O=CO", "carboxylic acid"),
("acetic acid", "CC(=O)O", "carboxylic acid"),
("propionic acid", "CCC(=O)O", "carboxylic acid"),
("butyric acid", "CCCC(=O)O", "carboxylic acid"),
("pentanoic acid", "CCCCC(=O)O", "carboxylic acid"),
("hexanoic acid", "CCCCCC(=O)O", "carboxylic acid"),
("heptanoic acid", "CCCCCCC(=O)O", "carboxylic acid"),
("octanoic acid", "CCCCCCCC(=O)O", "carboxylic acid"),
("lactic acid", "CC(O)C(=O)O", "carboxylic acid"),
("glycolic acid", "OCC(=O)O", "carboxylic acid"),
("benzene", "c1ccccc1", "aromatic"),
("toluene", "Cc1ccccc1", "aromatic"),
("ethylbenzene", "CCc1ccccc1", "aromatic"),
("propylbenzene", "CCCc1ccccc1", "aromatic"),
("xylene (o)", "Cc1ccccc1C", "aromatic"),
("xylene (m)", "Cc1cccc(C)c1", "aromatic"),
("xylene (p)", "Cc1ccc(C)cc1", "aromatic"),
("mesitylene", "Cc1cc(C)cc(C)c1", "aromatic"),
("naphthalene", "c1ccc2ccccc2c1", "aromatic"),
("biphenyl", "c1ccc(-c2ccccc2)cc1", "aromatic"),
]

In [ ]:
data = pd.DataFrame(records, columns=["name", "smiles", "family"]); y_ref = data.family.to_numpy(); N = len(data)
print(N, "molecules; families:", data.family.value_counts().to_dict())

## 1. Descriptors from SMILES, without RDKit
Eight numbers describe each molecule: molecular weight (MW), the Wildman--Crippen estimate of $\log P$, the topological polar surface area (TPSA), the numbers of hydrogen-bond donors (HBD) and acceptors (HBA), the number of rings, the fraction of sp$^3$ carbons, and the number of oxygen atoms. For molecules of C, H and O every one of them follows from the molecular graph by a short rule: MW from the atoms and hydrogens; **TPSA** as a sum over oxygen atoms ($20.23$ \AA$^2$ for a hydroxyl, $17.07$ for a carbonyl, $9.23$ for an ether); HBD the oxygens carrying a hydrogen; HBA the hydroxyl oxygens not attached to a carbonyl carbon plus the carbonyl and ether oxygens (so a carboxylic acid has $1$); rings from the cyclomatic number; and $\log P$ as a sum of **atom contributions** by atom type: $0.1441$ for a CH$_x$ carbon with carbon neighbors, $-0.2035$ for a CH$_2$ or CH$_3$ carbon bonded to oxygen, $-0.2051$ for a CH carbon bonded to oxygen, $0.1581$ for an aromatic CH, $0.1360$ for an aromatic carbon bearing a carbon substituent, $0.0845$ for a methyl on an aromatic ring, $0.123$ for each hydrocarbon hydrogen, $-0.2893$ and $-0.2677$ for the oxygen and hydrogen of a hydroxyl, and $-0.4222$ for a carboxyl group.
**Stop and predict:** the $\log P$ of methane ($1$ carbon, $4$ hydrogens) and of ethanol.

In [ ]:
import re
MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "S": 32.06, "F": 18.998, "Cl": 35.45, "Br": 79.904}
Z = {"H": 1, "C": 6, "N": 7, "O": 8, "F": 9, "S": 16, "Cl": 17, "Br": 35}
VAL = {"C": 4, "N": 3, "O": 2, "S": 2, "F": 1, "Cl": 1, "Br": 1}
TOKEN = re.compile(r"\[[^\]]+\]|Cl|Br|[BCNOSPFI]|[bcnosp]|\(|\)|=|#|-|:|\/|\\|\d|%\d\d|\.")
def parse_smiles(smi):
    """SMILES -> atoms (element, aromatic, charge, explicit_H or None) and bonds (i, j, order); order 1.5 marks an aromatic bond"""
    atoms, bonds, stack, prev, pending, rings = [], [], [], None, None, {}
    for tok in TOKEN.findall(smi):
        if tok == "(": stack.append(prev)
        elif tok == ")": prev = stack.pop()
        elif tok in "-=#:": pending = {"-": 1, "=": 2, "#": 3, ":": 1.5}[tok]
        elif tok.isdigit():
            if tok in rings: j, o = rings.pop(tok); bonds.append((prev, j, o if o is not None else (pending or (1.5 if atoms[prev][1] and atoms[j][1] else 1)))); pending = None
            else: rings[tok] = (prev, pending); pending = None
        else:
            if tok.startswith("["):
                m = re.match(r"\[([A-Za-z][a-z]?)(H\d?)?([+-]\d?)?\]", tok); el = m.group(1); arom = el.islower(); el = el.capitalize()
                nh = 0 if not m.group(2) else (1 if m.group(2) == "H" else int(m.group(2)[1:])); ch = 0 if not m.group(3) else (int(m.group(3)[1:] or 1) * (1 if m.group(3)[0] == "+" else -1)); atoms.append((el, arom, ch, nh))
            else: arom = tok.islower(); atoms.append((tok.capitalize(), arom, 0, None))
            i = len(atoms) - 1
            if prev is not None: bonds.append((prev, i, pending if pending else (1.5 if atoms[prev][1] and arom else 1)))
            prev, pending = i, None
    return atoms, bonds
def add_hydrogens(atoms, bonds):
    """implicit hydrogens from the default valences (aromatic bonds count 1, plus one pi bond for carbon and pyridine-type nitrogen)"""
    used = [0.0] * len(atoms); nb = [[] for _ in atoms]
    for i, j, o in bonds: nb[i].append((j, o)); nb[j].append((i, o)); w = 1 if o == 1.5 else o; used[i] += w; used[j] += w
    H = []
    for i, (el, arom, ch, nh) in enumerate(atoms):
        if nh is not None: H.append(nh); continue                                             # bracket atoms: the hydrogens are written
        v = VAL[el] + (1 if (el == "N" and ch == 1) else 0) - (1 if (el == "O" and ch == 1) else 0)
        u = used[i]
        if arom and el in ("C", "N") and u < v: u += 1                                          # the pi bond of an aromatic atom
        H.append(int(max(0, v - u)))
    return H, nb
def ring_atoms(n, bonds):
    """atoms and bonds that lie on a ring: a bond is a ring bond if its ends stay connected without it"""
    adj = {i: [] for i in range(n)}
    for k, (i, j, o) in enumerate(bonds): adj[i].append((j, k)); adj[j].append((i, k))
    def connected(a, b, skip):
        seen, st = {a}, [a]
        while st:
            x = st.pop()
            if x == b: return True
            for y, k in adj[x]:
                if k != skip and y not in seen: seen.add(y); st.append(y)
        return False
    rb = [connected(i, j, k) for k, (i, j, o) in enumerate(bonds)]; ra = set()
    for k, (i, j, o) in enumerate(bonds):
        if rb[k]: ra.update((i, j))
    return ra, rb
def mol_weight(atoms, H): return sum(MASS[a[0]] for a in atoms) + sum(H) * MASS["H"]

In [ ]:
CR = dict(C1=0.1441, C2=0.0, C3=-0.2035, C4=-0.2051, C5=-0.2783, C8=0.08452, C10=-0.0516, C11=0.1193, C18=0.1581, C19=0.2955, C20=0.2713, C21=0.1360)
H1, H2, O2, COOH = 0.1230, -0.2677, -0.2893, -0.4222        # hydrocarbon H; hydroxyl H and O; the carboxyl group (C, two O and the acid H), from acetic, propionic and butyric acid
def descriptors(smi):
    at, bd = parse_smiles(smi); H, nb_ = add_hydrogens(at, bd); n = len(at); el = [a[0] for a in at]; arom = [a[1] for a in at]
    carbonylC = [el[i] == "C" and any(el[j] == "O" and o == 2 for j, o in nb_[i]) for i in range(n)]
    cooh = [i for i in range(n) if carbonylC[i] and any(el[j] == "O" and o == 1 and H[j] > 0 for j, o in nb_[i])]
    acid = set(cooh) | {j for i in cooh for j, o in nb_[i] if el[j] == "O"}; logp = COOH * len(cooh)
    for i in range(n):
        if i in acid: continue
        h = H[i]; nbs = [j for j, o in nb_[i]]
        if el[i] == "O": logp += O2 + H2 * h
        elif el[i] == "C":
            if arom[i]:
                other = [j for j, o in nb_[i] if o != 1.5]; t = "C18" if h == 1 else ("C19" if not other else ("C20" if arom[other[0]] else "C21"))
            elif any(el[j] == "O" for j in nbs): t = "C3" if h >= 2 else ("C4" if h == 1 else "C5")
            elif any(arom[j] for j in nbs): t = "C8" if h == 3 else ("C10" if h == 2 else "C11")
            else: t = "C1" if h >= 2 else "C2"
            logp += CR[t] + H1 * h
    logp += H1 * sum(H[i] for i in cooh)                                         # a formyl hydrogen on the carboxyl carbon
    Os = [i for i in range(n) if el[i] == "O"]
    hba = sum(1 for i in Os if any(o == 2 for j, o in nb_[i]) or (H[i] > 0 and not any(carbonylC[j] for j, o in nb_[i])) or (H[i] == 0 and len(nb_[i]) == 2))
    tpsa = sum(17.07 if any(o == 2 for j, o in nb_[i]) else (20.23 if H[i] > 0 else 9.23) for i in Os)
    cs = [i for i in range(n) if el[i] == "C"]; sp3 = [i for i in cs if not arom[i] and all(o == 1 for j, o in nb_[i])]
    return dict(MW=mol_weight(at, H), logP=logp, TPSA=tpsa, HBD=sum(H[i] > 0 for i in Os), HBA=hba, rings=len(bd) - n + 1, fraction_Csp3=len(sp3) / len(cs), oxygen_atoms=len(Os))
features = pd.DataFrame([descriptors(s) for s in data.smiles]); X_raw = features.to_numpy(float); feat_names = list(features.columns)
print(features.head(5).round(4).to_string())
rdkit_logp = {"C": .6361, "CC": 1.0262, "CCC": 1.4163, "CCCC": 1.8064, "CCCCC": 2.1965, "CCCCCC": 2.5866, "CCCCCCC": 2.9767, "CCCCCCCC": 3.3668, "CCO": -.0014, "CCCO": .3887, "CCCCO": .7788, "CC(C)O": .3871, "CO": -.3915,
              "CC(=O)O": .0909, "CCC(=O)O": .4810, "CCCC(=O)O": .8711, "c1ccccc1": 1.6866, "Cc1ccccc1": 1.99502, "CCc1ccccc1": 2.2490, "c1ccc2ccccc2c1": 2.8398, "c1ccc(-c2ccccc2)cc1": 3.3536, "OCCO": -1.0290, "C1CCCCC1": 2.3406}
dlp = max(abs(descriptors(s)["logP"] - v) for s, v in rdkit_logp.items())
print(f"\nlogP of {len(rdkit_logp)} of the molecules against RDKit's values (printed in the notebook's first rows and in the table of Post A6b): largest difference {dlp:.1e}")
print("methane: 0.1441 + 4 x 0.123 =", round(0.1441 + 4 * 0.123, 4), "| ethanol: CH3 0.1441 + CH2(O) -0.2035 + OH (-0.2893 - 0.2677) + 5 H 0.615 =", round(0.1441 - 0.2035 - 0.2893 - 0.2677 + 5 * 0.123, 4))
assert dlp < 1e-4 and features.shape == (40, 8) and np.allclose(features.iloc[0, :2], [16.043, 0.6361], atol=1e-3)

## 2. The stratified 60/20/20 split
The notebook splits the molecules 60/20/20 into training, validation and test sets with the seed 42, *stratified* by family so that each set has all four families. The first split takes $40\%$ of the molecules aside, the second divides those in half. We rebuild scikit-learn's procedure: per family, a random permutation of its members (from `RandomState(42)`); the first $n_{\rm train}$ go to the training set, the next $n_{\rm test}$ to the other set; the two sets are then shuffled.
**Stop and predict:** how many molecules of each family are in each set?

In [ ]:
def approx_mode(counts, n_draws, rs):
    cont = counts / counts.sum() * n_draws; fl = np.floor(cont); need = int(n_draws - fl.sum())
    if need > 0:
        rem = cont - fl
        for v in np.sort(np.unique(rem))[::-1]:
            ind = np.where(rem == v)[0]; add = min(len(ind), need); fl[rs.choice(ind, size=add, replace=False)] += 1; need -= add
            if need == 0: break
    return fl.astype(int)
def stratified_split(idx, labels, test_size, seed):
    """train_test_split(idx, test_size=..., random_state=seed, stratify=labels), written out"""
    rs = np.random.RandomState(seed); n = len(idx); n_test = int(np.ceil(test_size * n)); n_train = n - n_test
    cls, yi = np.unique(labels, return_inverse=True); cc = np.bincount(yi); ci = np.split(np.argsort(yi, kind="mergesort"), np.cumsum(cc)[:-1])
    n_i = approx_mode(cc, n_train, rs); t_i = approx_mode(cc - n_i, n_test, rs); tr, te = [], []
    for i in range(len(cls)):
        perm = rs.permutation(cc[i]); pc = ci[i].take(perm, mode="clip"); tr.extend(pc[:n_i[i]]); te.extend(pc[n_i[i]:n_i[i] + t_i[i]])
    return idx[rs.permutation(tr)], idx[rs.permutation(te)]
ids = np.arange(N); train_ids, other_ids = stratified_split(ids, y_ref, 0.40, SEED); val_ids, test_ids = stratified_split(other_ids, y_ref[other_ids], 0.50, SEED)
print("train / validation / test:", len(train_ids), len(val_ids), len(test_ids), "(the notebook prints 24, 8, 8)")
print(pd.DataFrame({"train": pd.Series(y_ref[train_ids]).value_counts(), "validation": pd.Series(y_ref[val_ids]).value_counts(), "test": pd.Series(y_ref[test_ids]).value_counts()}).to_string())
print("test molecules:", list(data.name[test_ids]))
assert (len(train_ids), len(val_ids), len(test_ids)) == (24, 8, 8) and set(data.name[test_ids]) == {"octanoic acid", "biphenyl", "glycerol", "pentane", "2-propanol", "acetic acid", "ethylbenzene", "hexane"}

## 3. Standardization and PCA
Euclidean distances treat a difference of 30 in molecular weight and of one ring alike unless the columns are rescaled: **standardization** subtracts the training mean and divides by the training standard deviation (the notebook's example: $30,40,50$ has mean $40$, standard deviation $8.16$, and $50$ becomes $1.225$). **PCA** rotates the standardized data to new axes of decreasing variance; it is the singular value decomposition $Z=U\Sigma V^T$ of the centred training matrix, the rows of $V^T$ are the *loadings*, the variance along component $k$ is $\sigma_k^2/(n-1)$, and the sign of a component is fixed by making its largest loading positive. Both are fitted on the **training molecules only** and then applied to the others, so that nothing about the validation and test molecules leaks into them.
**Stop and predict:** what share of the variance do the first two components carry?

In [ ]:
class StandardScaler:
    def fit(self, X): self.mean_, self.scale_ = X.mean(0), X.std(0); return self
    def transform(self, X): return (X - self.mean_) / self.scale_
class PCA:
    def __init__(self, n): self.n = n
    def fit(self, X):
        U, S, Vt = np.linalg.svd(X - X.mean(0), full_matrices=False); Vt = Vt * np.sign(Vt[np.arange(len(Vt)), np.abs(Vt).argmax(1)])[:, None]            # the largest loading of each component is positive
        self.mean_, self.components_, var = X.mean(0), Vt[:self.n], S ** 2 / (len(X) - 1); self.explained_variance_ratio_ = var[:self.n] / var.sum(); return self
    def transform(self, X): return (X - self.mean_) @ self.components_.T
w = np.array([30., 40., 50.]); print(f"the notebook's example: mean {w.mean():.2f}, standard deviation {w.std():.2f}, 50 becomes {(50 - w.mean()) / w.std():.3f}")
scaler = StandardScaler().fit(X_raw[train_ids]); X_train, X_val, X_test = (scaler.transform(X_raw[i]) for i in (train_ids, val_ids, test_ids))
print("means of the standardized training columns:", np.round(X_train.mean(0), 3))
pca = PCA(2).fit(X_train); Z_train = pca.transform(X_train)
print("variance explained by PC1 and PC2:", pca.explained_variance_ratio_.round(3), "| together", round(pca.explained_variance_ratio_.sum(), 3), "(the notebook prints 0.632, 0.215 and 0.847)")
print(pd.DataFrame(pca.components_.T, index=feat_names, columns=["PC1 loading", "PC2 loading"]).round(3).to_string())
assert np.allclose(pca.explained_variance_ratio_, [0.632, 0.215], atol=5e-4) and np.allclose(pca.components_[0], [-0.135, -0.386, 0.417, 0.429, 0.429, -0.313, 0.149, 0.413], atol=1e-3) and abs((50 - w.mean()) / w.std() - 1.225) < 1e-3

## 4. K-means, silhouette, and the choice of $K$
**K-means** repeats two steps: assign every molecule to the nearest centroid; move each centroid to the mean of its molecules; it minimizes the **inertia** $J=\sum_i\|x_i-\mu_{c_i}\|^2$. The result depends on the starting centroids, so the algorithm is run from several starts (with the $k$-means++ rule of choosing starts far apart) and the best is kept. The notebook uses $20$ starts; our random numbers are not scikit-learn's, so we use $100$, and we show why below. The **silhouette** of a molecule is $(b-a)/\max(a,b)$ with $a$ its mean distance to its own cluster and $b$ to the nearest other cluster; its mean runs from $-1$ to $1$. The inertia always falls as $K$ grows, so $K$ is chosen by the silhouette of the **validation** molecules, which are assigned to the training centroids.
**Stop and predict:** which $K$ will the validation silhouette choose, and how many chemical families will it find?

In [ ]:
def kmeans_pp(X, k, rs):
    c = [X[rs.randint(len(X))]]
    for _ in range(1, k): d2 = np.min([((X - ci) ** 2).sum(1) for ci in c], axis=0); c.append(X[rs.choice(len(X), p=d2 / d2.sum())])
    return np.array(c)
def kmeans(X, k, n_init=100, seed=SEED):
    rs = np.random.RandomState(seed); best = None
    for _ in range(n_init):
        C = kmeans_pp(X, k, rs)
        for _ in range(300):
            lab = ((X[:, None] - C[None]) ** 2).sum(-1).argmin(1); Cn = np.array([X[lab == j].mean(0) if (lab == j).any() else C[j] for j in range(k)])
            if np.allclose(Cn, C): break
            C = Cn
        lab = ((X[:, None] - C[None]) ** 2).sum(-1).argmin(1); J = sum(((X[lab == j] - C[j]) ** 2).sum() for j in range(k))
        if best is None or J < best[0] - 1e-12: best = (J, C, lab)
    return best
def assign(X, C): return ((X[:, None] - C[None]) ** 2).sum(-1).argmin(1)
def silhouette(X, lab):
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1)); s = np.zeros(len(X)); ks = np.unique(lab)
    for i in range(len(X)):
        same = (lab == lab[i]); same[i] = False
        if not same.any(): continue
        a = D[i, same].mean(); b = min(D[i, lab == k_].mean() for k_ in ks if k_ != lab[i]); s[i] = (b - a) / max(a, b)
    return s.mean()
J6_20 = kmeans(X_train, 6, n_init=20)[0]; J6_100 = kmeans(X_train, 6)[0]
print(f"local optima: K = 6 with 20 of our starts gives inertia {J6_20:.3f}, with 100 starts {J6_100:.3f} (scikit-learn's 20 starts: 24.255)")
J4, C4, lab4 = kmeans(X_train, 4); print("K = 4 on the training molecules: group sizes", sorted(np.bincount(lab4)), "inertia", round(J4, 3), "(the notebook prints sizes 4, 6, 6, 8 and 44.55)")
rows = []
for k in (2, 3, 4, 5, 6):
    J, C, lab = kmeans(X_train, k); vl = assign(X_val, C); ok = 1 < len(set(vl)) < len(vl)
    rows.append((k, J, silhouette(X_train, lab), silhouette(X_val, vl) if ok else np.nan, len(set(vl))))
search = pd.DataFrame(rows, columns=["K", "train_inertia", "train_silhouette", "validation_silhouette", "clusters occupied by the 8 validation molecules"]); print(search.round(3).to_string(index=False))
print("validation silhouettes with more digits, K = 3 and K = 4:", search.validation_silhouette.iloc[1], search.validation_silhouette.iloc[2], "| equal:", search.validation_silhouette.iloc[1] == search.validation_silhouette.iloc[2])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5)); ax[0].plot(search.K, search.train_inertia, "o-", color=COLORS["blue"]); ax[0].set(xlabel="K", ylabel="inertia (training)", title="(a) elbow plot")
ax[1].plot(search.K, search.train_silhouette, "o-", color=COLORS["blue"], label="training"); ax[1].plot(search.K, search.validation_silhouette, "s--", color=COLORS["orange"], label="validation (8 molecules)"); ax[1].set(xlabel="K", ylabel="silhouette", title="(b) comparing the candidates"); ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()
best_k = int(search.dropna().loc[search.dropna().validation_silhouette.idxmax(), "K"]); print("K chosen by the validation silhouette:", best_k, "(the notebook: 3)")
print(f"the validation silhouettes of K = 3 and K = 4 are identical ({search.validation_silhouette.iloc[1]:.6f}): at K = 4 the validation molecules occupy only {search.iloc[2, 4]} of the 4 clusters, the same partition as at K = 3, so the validation set cannot tell K = 3 from K = 4 and the first is kept by the tie")
assert search.validation_silhouette.iloc[1] == search.validation_silhouette.iloc[2] and search.iloc[2, 4] == 3
assert J6_20 > J6_100 + 0.1 and np.allclose(search.train_inertia, [94.040, 64.134, 44.550, 33.742, 24.255], atol=5e-4) and np.allclose(search.train_silhouette, [.427, .436, .428, .442, .479], atol=5e-4) and np.allclose(search.validation_silhouette, [.540, .566, .566, .244, .197], atol=5e-4) and best_k == 3

## 5. ARI, the post-hoc mapping, and what the families look like
After the clustering, the withheld family labels give an **external check**. The **adjusted Rand index** (ARI) compares two partitions through the pairs of molecules they put together: it is $1$ for identical partitions and about $0$ for chance agreement, and it does not need the cluster numbers to match the family names. To turn clusters into predicted *names*, a majority vote of the fit molecules (training and validation) names each cluster; this is a supervised step, and the resulting accuracy is not an unsupervised measure. The final model is fitted on the **fit** set (training and validation) and applied to the eight test molecules.
**Stop and predict:** which families can $K=3$ clusters separate, and which will be merged?

In [ ]:
def ari(a, b):
    ca, cb = np.unique(a, return_inverse=True)[1], np.unique(b, return_inverse=True)[1]; M = np.zeros((ca.max() + 1, cb.max() + 1))
    for i, j in zip(ca, cb): M[i, j] += 1
    c2 = lambda x: x * (x - 1) / 2; sij = c2(M).sum(); sa, sb = c2(M.sum(1)).sum(), c2(M.sum(0)).sum(); exp = sa * sb / c2(len(a)); return (sij - exp) / (0.5 * (sa + sb) - exp)
fit_ids = np.concatenate([train_ids, val_ids]); scaler2 = StandardScaler().fit(X_raw[fit_ids]); X_fit, X_hold = scaler2.transform(X_raw[fit_ids]), scaler2.transform(X_raw[test_ids])
Jf, Cf, clusters_fit = kmeans(X_fit, best_k); clusters_test = assign(X_hold, Cf)
summ = features.iloc[fit_ids].copy(); summ["cluster"] = clusters_fit; order = summ.groupby("cluster").MW.mean().rank().astype(int).sub(1)
print("mean descriptors of the fit molecules in each cluster (clusters listed by increasing molecular weight):"); print(summ.groupby("cluster").mean().loc[order.sort_values().index].round(2).to_string())
a_ = ari(y_ref[test_ids], clusters_test); print(f"\nARI of the 8 test molecules against their families: {a_:.3f} (the notebook prints 0.588)")
print(pd.crosstab(pd.Series(y_ref[test_ids], name="reference family"), pd.Series(clusters_test, name="K-means cluster")).to_string())
mapping = {int(c): pd.Series(y_ref[fit_ids][clusters_fit == c]).mode()[0] for c in np.unique(clusters_fit)}; pred = np.array([mapping[int(c)] for c in clusters_test]); acc = (pred == y_ref[test_ids]).mean()
print("majority-vote mapping:", mapping, "| post-hoc test accuracy", acc, "(the notebook prints 0.75: the two acids are named alcohol)")
assert abs(a_ - 0.588) < 5e-4 and acc == 0.75 and sorted(mapping.values()) == ["alcohol", "aromatic", "hydrocarbon"]

In [ ]:
Xall = StandardScaler().fit(X_raw).transform(X_raw); pca_all = PCA(2).fit(Xall); Zall = pca_all.transform(Xall); rows = []
for k in (2, 3, 4, 5):
    J, C, lab = kmeans(Xall, k); rows.append((k, J, silhouette(Xall, lab), ari(y_ref, lab)))
full = pd.DataFrame(rows, columns=["K", "inertia", "silhouette", "ARI against the 4 families"]); print("all 40 molecules (no splitting; a descriptive view):\n" + full.round(3).to_string(index=False))
J4a, C4a, lab4a = kmeans(Xall, 4); print("\nfamilies against the 4 clusters:\n" + pd.crosstab(pd.Series(y_ref, name="family"), pd.Series(lab4a, name="cluster")).to_string())
print("acids and alcohols overlap in descriptor space: TPSA of the acids", sorted(set(features.TPSA[data.family == "carboxylic acid"].round(2))), "against the alcohols", sorted(set(features.TPSA[data.family == "alcohol"].round(2))), "| glycerol and ethylene glycol lie among the acids")
fig, ax = plt.subplots(1, 2, figsize=(11, 4)); fam_c = {"hydrocarbon": COLORS["blue"], "alcohol": COLORS["orange"], "carboxylic acid": COLORS["red"], "aromatic": COLORS["green"]}
for f_, c_ in fam_c.items(): m = y_ref == f_; ax[0].scatter(Zall[m, 0], Zall[m, 1], c=c_, s=55, label=f_, edgecolor="k", linewidth=0.3)
ax[0].set(xlabel="PC1", ylabel="PC2", title="(a) the 40 molecules by family"); ax[0].legend(fontsize=8)
ax[1].scatter(Zall[:, 0], Zall[:, 1], c=lab4a, cmap="tab10", s=55, edgecolor="k", linewidth=0.3); ax[1].set(xlabel="PC1", ylabel="PC2", title="(b) K-means, K = 4 (cluster numbers are arbitrary)"); plt.tight_layout(); plt.show()
assert full.iloc[:, 3].max() > 0.5

## 6. Random forests: supervised, and the proximity construction
A **decision tree** splits the molecules again and again by a threshold on one feature, choosing the split that most reduces the Gini impurity $1-\sum_cp_c^2$, and stops when the leaves are pure or too small. A **random forest** grows many trees, each on a bootstrap resample of the molecules and with only $\lfloor\sqrt{p}\rfloor=2$ randomly chosen features considered at each split, and averages their votes. This is *supervised*: it needs the family labels. The **proximity** construction uses no labels: add a shuffled copy of the data (each column permuted independently, which destroys the correlations between descriptors), train a forest to tell real from shuffled, and call two real molecules *similar* when they land in the same leaf in many trees; the dissimilarity $1-\text{proximity}$ is clustered hierarchically (average linkage). Our forest is our own, so its random numbers are not scikit-learn's: the numbers will differ in detail from the notebook's; Part II runs scikit-learn's forest on the same table.
**Stop and predict:** the validation accuracy of a forest on 8 molecules can only take which values?

In [ ]:
class Tree:
    def __init__(self, min_leaf=1, max_features=None, rs=None): self.min_leaf, self.mf, self.rs = min_leaf, max_features, rs
    def _gini(self, y): p = np.bincount(y, minlength=self.nc) / len(y); return 1 - (p ** 2).sum()
    def _grow(self, X, y, depth=0):
        node = {"leaf": len(self.nodes), "p": np.bincount(y, minlength=self.nc) / len(y)}; self.nodes.append(node)
        if self._gini(y) == 0 or len(y) < 2 * self.min_leaf: return node
        feats = self.rs.permutation(X.shape[1]); best = None; tried = 0
        for f in feats:
            if tried >= self.mf and best is not None: break
            tried += 1; v = np.unique(X[:, f])
            for t in (v[:-1] + v[1:]) / 2:
                L = X[:, f] <= t
                if L.sum() < self.min_leaf or (~L).sum() < self.min_leaf: continue
                g = (L.sum() * self._gini(y[L]) + (~L).sum() * self._gini(y[~L])) / len(y)
                if best is None or g < best[0] - 1e-12: best = (g, f, t)
        if best is None: return node
        _, f, t = best; L = X[:, f] <= t; node.update(f=f, t=t); node["l"] = self._grow(X[L], y[L], depth + 1); node["r"] = self._grow(X[~L], y[~L], depth + 1); return node
    def fit(self, X, y, nc): self.nc, self.nodes = nc, []; self.root = self._grow(X, y); return self
    def _leaf(self, x):
        n = self.root
        while "f" in n: n = n["l"] if x[n["f"]] <= n["t"] else n["r"]
        return n
    def predict_proba(self, X): return np.array([self._leaf(x)["p"] for x in X])
    def apply(self, X): return np.array([self._leaf(x)["leaf"] for x in X])
class Forest:
    def __init__(self, n_trees=100, min_leaf=1, max_features=None, seed=0): self.n, self.leaf, self.mf, self.seed = n_trees, min_leaf, max_features, seed
    def fit(self, X, y):
        self.classes_, yi = np.unique(y, return_inverse=True); rs = np.random.RandomState(self.seed); mf = self.mf or max(1, int(math.sqrt(X.shape[1]))); self.trees = []
        for _ in range(self.n): b = rs.randint(0, len(X), len(X)); self.trees.append(Tree(self.leaf, mf, rs).fit(X[b], yi[b], len(self.classes_)))
        return self
    def predict_proba(self, X): return np.mean([t.predict_proba(X) for t in self.trees], axis=0)
    def predict(self, X): return self.classes_[self.predict_proba(X).argmax(1)]
    def apply(self, X): return np.stack([t.apply(X) for t in self.trees], 1)
acc_val = lambda m, Xa, ya: float((m.predict(Xa) == ya).mean())
t0 = time.perf_counter(); rows = []
for nt, leaf in [(30, 2), (100, 2), (100, 4), (200, 2)]:
    m = Forest(nt, leaf, seed=SEED).fit(X_train, y_ref[train_ids]); rows.append((nt, leaf, acc_val(m, X_val, y_ref[val_ids])))
print(pd.DataFrame(rows, columns=["n_trees", "min_samples_leaf", "validation accuracy (our forest)"]).to_string(index=False), f"[{time.perf_counter() - t0:.0f} s]")
print("with 8 validation molecules an accuracy is a multiple of 0.125; the notebook's table (scikit-learn's forest) reads 1.000, 0.875, 1.000, 0.875: a difference of one molecule")
rf = Forest(100, 2, seed=SEED).fit(X_fit, y_ref[fit_ids]); print("our forest on the 8 test molecules: accuracy", acc_val(rf, X_hold, y_ref[test_ids]), "| errors:", [(data.name[i], p) for i, p in zip(test_ids, rf.predict(X_hold)) if p != y_ref[i]])
rs0 = np.random.RandomState(SEED); X_synth = np.column_stack([rs0.permutation(X_fit[:, j]) for j in range(X_fit.shape[1])]); Xb = np.vstack([X_fit, X_synth]); yb = np.r_[np.ones(len(X_fit), int), np.zeros(len(X_synth), int)]
urf = Forest(150, 2, seed=SEED).fit(Xb, yb); lv = urf.apply(X_hold); prox = (lv[:, None, :] == lv[None, :, :]).mean(2); dist = 1 - prox; np.fill_diagonal(dist, 0)
print("proximity of the test molecules: mean off-diagonal", round(prox[~np.eye(8, dtype=bool)].mean(), 3), "| the pair with the highest proximity:", max(((prox[i, j], data.name[test_ids[i]], data.name[test_ids[j]]) for i in range(8) for j in range(i + 1, 8))))
assert set(r[2] for r in rows) <= {k / 8 for k in range(9)} and prox.shape == (8, 8) and np.allclose(np.diag(prox), 1)

In [ ]:
def average_linkage(D, k):
    cl = [[i] for i in range(len(D))]
    while len(cl) > k:
        best = None
        for a in range(len(cl)):
            for b in range(a + 1, len(cl)):
                d = np.mean([D[i, j] for i in cl[a] for j in cl[b]])
                if best is None or d < best[0] - 1e-12: best = (d, a, b)
        _, a, b = best; cl[a] = cl[a] + cl[b]; del cl[b]
    lab = np.zeros(len(D), int)
    for c, members in enumerate(cl): lab[members] = c
    return lab
urf_clusters = average_linkage(dist, best_k); print("our proximity clusters of the test molecules:", urf_clusters.tolist(), "| ARI against the families:", round(ari(y_ref[test_ids], urf_clusters), 3), "(scikit-learn's forest gave 0.329 in the notebook)")
print("pairs together by the family labels:", sum(y_ref[test_ids][i] == y_ref[test_ids][j] for i in range(8) for j in range(i + 1, 8)), "of 28")

## 7. How much can 8 test molecules say?
The notebook ends with the caution that eight test molecules cannot support precise claims. We measure it. **(a)** The same forest with 30 different random seeds: how much does the validation accuracy of each setting of the notebook's table vary, and does the setting matter? **(b)** The ARI of the proximity clustering over 30 seeds against the ARI of K-means, which has no randomness left at this size. **(c)** A bootstrap interval for the K-means accuracy and ARI of the test set.
**Stop and predict:** do the four forest settings differ by more than the seed does?

In [ ]:
t0 = time.perf_counter(); seeds = range(30); res = {}
for nt, leaf in [(30, 2), (100, 2), (100, 4), (200, 2)]:
    res[(nt, leaf)] = [acc_val(Forest(nt if nt <= 100 else 100, leaf, seed=s).fit(X_train, y_ref[train_ids]), X_val, y_ref[val_ids]) for s in seeds]
tab = pd.DataFrame([(f"{nt} trees, leaf {lf}" + (" (100 used)" if nt > 100 else ""), np.mean(v), np.std(v), min(v), max(v)) for (nt, lf), v in res.items()], columns=["setting", "mean validation accuracy", "sd over 30 seeds", "min", "max"]); print(tab.round(3).to_string(index=False))
ua, tacc = [], []
for s in seeds:
    u = Forest(60, 2, seed=s).fit(Xb, yb); lv = u.apply(X_hold); pr = (lv[:, None, :] == lv[None, :, :]).mean(2); d = 1 - pr; np.fill_diagonal(d, 0); ua.append(ari(y_ref[test_ids], average_linkage(d, best_k)))
    tacc.append(acc_val(Forest(100, 2, seed=s).fit(X_fit, y_ref[fit_ids]), X_hold, y_ref[test_ids]))
print("seeds (of 30) whose validation accuracy is 0.875 instead of 1.0, per setting:", dict(zip(["30/2", "100/2", "100/4", "200/2"], [int(sum(a < 1 for a in res[k_])) for k_ in res])))
print(f"\nproximity clustering, ARI over 30 seeds: {np.mean(ua):.3f} +- {np.std(ua):.3f} (range {min(ua):.3f} to {max(ua):.3f}) against K-means {a_:.3f}; supervised forest test accuracy over 30 seeds: {np.mean(tacc):.3f} +- {np.std(tacc):.3f} (range {min(tacc):.3f} to {max(tacc):.3f}) [{time.perf_counter() - t0:.0f} s]")
rg = np.random.default_rng(3); ba, bacc = [], []; yt = y_ref[test_ids]
for _ in range(2000):
    i = rg.integers(0, 8, 8)
    if len(set(clusters_test[i])) > 1 and len(set(yt[i])) > 1: ba.append(ari(yt[i], clusters_test[i]))
    bacc.append((pred[i] == yt[i]).mean())
print(f"bootstrap over the 8 test molecules (K-means): accuracy {np.mean(bacc):.2f}, 95% interval [{np.percentile(bacc, 2.5):.2f}, {np.percentile(bacc, 97.5):.2f}]; ARI 95% interval [{np.percentile(ba, 2.5):.2f}, {np.percentile(ba, 97.5):.2f}] ({len(ba)} of 2000 resamples defined)")
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.7)); n_low = [int(sum(a < 1 for a in res[k_])) for k_ in res]; ax[0].bar(["30/2", "100/2", "100/4", "200/2"], n_low, color=COLORS["blue"]); ax[0].set(xlabel="trees / min leaf", ylabel="seeds (of 30) with accuracy 0.875 instead of 1.0", title="(a) the setting hardly matters; the seed does")
ax[1].hist(ua, bins=np.linspace(-0.2, 1, 13), color=COLORS["orange"], alpha=0.8, label="proximity clustering, 30 seeds"); ax[1].axvline(a_, color=COLORS["blue"], ls="--", label=f"K-means {a_:.3f}"); ax[1].set(xlabel="ARI on the 8 test molecules", ylabel="seeds", title="(b) the proximity construction against K-means"); ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()
assert tab.iloc[:, 2].max() > 0.0 and np.std(ua) > 0.05

## 8. Your turn
**Exercise 1.** Cluster with only MW and $\log P$, then with only the three polar descriptors (TPSA, HBD, HBA). Which families separate?
**Exercise 2.** Add a ninth descriptor, the number of carbons. How do the silhouettes and the clusters change?
**Exercise 3.** Repeat Section 4 with a different seed for the split. Is the selected $K$ stable?
**Exercise 4.** Add a molecule that belongs to two families (4-hydroxybenzoic acid). Where does K-means put it, and where does the forest?
**Exercise 5.** Replace K-means by a Gaussian mixture or DBSCAN and say when it would be preferable.

## 9. Self-check

In [ ]:
assert dlp < 1e-4 and np.allclose(pca.explained_variance_ratio_, [0.632, 0.215], atol=5e-4) and np.allclose(search.train_inertia, [94.040, 64.134, 44.550, 33.742, 24.255], atol=5e-4)
assert best_k == 3 and abs(a_ - 0.588) < 5e-4 and acc == 0.75 and np.std(ua) > 0.05
print("All checks passed: you reproduced the numbers in the post.")

# Part II. Cross-check with scikit-learn and RDKit
> **Read this first.** Part II compares the functions of Part I with scikit-learn (preinstalled in Colab) and RDKit (installed by the cell if it is missing): the split, PCA, K-means (inertia), silhouette and ARI; scikit-learn's own forest run on our descriptor table, which reproduces the notebook's table, test accuracy and proximity ARI; and, with RDKit, all eight descriptors of all 40 molecules against ours. Each stage is guarded; the last cell lists the checks that passed.

In [ ]:
HAVE_SK = importlib.util.find_spec("sklearn") is not None; checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
if HAVE_SK:
    import warnings; warnings.filterwarnings("ignore"); import sklearn
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler as SkScaler
    from sklearn.decomposition import PCA as SkPCA
    from sklearn.cluster import KMeans, AgglomerativeClustering
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import silhouette_score, adjusted_rand_score, accuracy_score
    print("scikit-learn", sklearn.__version__)
    try:
        a1, b1 = train_test_split(ids, test_size=.40, random_state=SEED, stratify=y_ref); c1, d1 = train_test_split(b1, test_size=.50, random_state=SEED, stratify=y_ref[b1])
        check("the stratified split equals train_test_split (training, validation and test molecules, in the same order)", np.array_equal(a1, train_ids) and np.array_equal(c1, val_ids) and np.array_equal(d1, test_ids))
        sc_ = SkScaler().fit(X_raw[train_ids]); check("standardization equals StandardScaler", np.allclose(sc_.transform(X_raw[train_ids]), X_train))
        p_ = SkPCA(2).fit(X_train); check("PCA: loadings (with signs) and explained variance equal scikit-learn's", np.allclose(p_.components_, pca.components_, atol=1e-8) and np.allclose(p_.explained_variance_ratio_, pca.explained_variance_ratio_))
        ok_in = all(abs(KMeans(k, n_init=20, random_state=SEED).fit(X_train).inertia_ - search.train_inertia[k - 2]) < 1e-6 for k in (2, 3, 4, 5, 6)); check("K-means: our inertia equals KMeans(n_init=20) for K = 2 to 6", ok_in)
        km_ = KMeans(3, n_init=20, random_state=SEED).fit(X_train); J3, C3, lab3 = kmeans(X_train, 3); check("silhouette and ARI equal scikit-learn's", abs(silhouette_score(X_train, lab3) - silhouette_score(X_train, km_.labels_)) < 1e-12 and abs(ari(y_ref[train_ids], lab3) - adjusted_rand_score(y_ref[train_ids], km_.labels_)) < 1e-12)
    except Exception as e: print("stage could not be completed:", type(e).__name__, e)
else: print("scikit-learn is not available: install it and re-run this cell.")

In [ ]:
if HAVE_SK:
    try:
        rows_ = []
        for nt, leaf in [(30, 2), (100, 2), (100, 4), (200, 2)]: rows_.append(accuracy_score(y_ref[val_ids], RandomForestClassifier(n_estimators=nt, min_samples_leaf=leaf, random_state=SEED).fit(X_train, y_ref[train_ids]).predict(X_val)))
        print("scikit-learn's forest on our table, validation accuracy:", rows_, "| the notebook: [1.0, 0.875, 1.0, 0.875]")
        check("the notebook's random-forest validation table is reproduced (scikit-learn's forest on our descriptor table; may differ with the scikit-learn version)", np.allclose(rows_, [1.0, 0.875, 1.0, 0.875]))
        ch = sorted(zip(rows_, [-30, -100, -100, -200], [2, 2, 4, 2]), reverse=True)[0]; nt_, lf_ = -ch[1], ch[2]
        skrf = RandomForestClassifier(n_estimators=nt_, min_samples_leaf=lf_, random_state=SEED).fit(X_fit, y_ref[fit_ids]); a_rf = accuracy_score(y_ref[test_ids], skrf.predict(X_hold)); print("selected setting", nt_, lf_, "| test accuracy", a_rf, "(the notebook: 0.875)")
        check("the notebook's supervised forest test accuracy 0.875 is reproduced", abs(a_rf - 0.875) < 1e-9)
        print("the molecule that scikit-learn's forest gets wrong:", [(data.name[i], "predicted " + p, "family " + y_ref[i]) for i, p in zip(test_ids, skrf.predict(X_hold)) if p != y_ref[i]])
        rs_ = np.random.default_rng(SEED); Xs_ = np.column_stack([rs_.permutation(X_fit[:, j]) for j in range(X_fit.shape[1])]); rp = RandomForestClassifier(n_estimators=150, min_samples_leaf=2, max_features="sqrt", random_state=SEED).fit(np.vstack([X_fit, Xs_]), np.r_[np.ones(len(X_fit), int), np.zeros(len(Xs_), int)])
        lv_ = rp.apply(X_hold); pr_ = (lv_[:, None, :] == lv_[None, :, :]).mean(2); d_ = 1 - pr_; np.fill_diagonal(d_, 0); uc = AgglomerativeClustering(n_clusters=best_k, metric="precomputed", linkage="average").fit_predict(d_); a_u = adjusted_rand_score(y_ref[test_ids], uc)
        print("proximity clustering with scikit-learn's forest: ARI", round(a_u, 3), "(the notebook: 0.329) | our average linkage on the same distances gives the same partition:", adjusted_rand_score(uc, average_linkage(d_, best_k)) == 1.0)
        ua_sk = []
        for sd in range(30):
            rp_ = RandomForestClassifier(n_estimators=150, min_samples_leaf=2, max_features="sqrt", random_state=sd).fit(np.vstack([X_fit, Xs_]), np.r_[np.ones(len(X_fit), int), np.zeros(len(Xs_), int)]); l_ = rp_.apply(X_hold); q_ = 1 - (l_[:, None, :] == l_[None, :, :]).mean(2); np.fill_diagonal(q_, 0)
            ua_sk.append(adjusted_rand_score(y_ref[test_ids], AgglomerativeClustering(n_clusters=best_k, metric="precomputed", linkage="average").fit_predict(q_)))
        print(f"scikit-learn's forest, proximity ARI over 30 seeds (the shuffled copy fixed): {np.mean(ua_sk):.3f} +- {np.std(ua_sk):.3f} (range {min(ua_sk):.3f} to {max(ua_sk):.3f}); K-means: 0.588 | the notebook's seed 42 gives 0.329")
        check("the proximity ARI varies with the seed of the forest: the notebook's 0.329 lies inside the range of 30 seeds", min(ua_sk) - 1e-9 <= 0.329 <= max(ua_sk) + 1e-9 and np.std(ua_sk) > 0.03)
        check("the notebook's proximity ARI 0.329 is reproduced", abs(a_u - 0.329) < 5e-4); check("our average-linkage clustering equals AgglomerativeClustering on the same distance matrix", adjusted_rand_score(uc, average_linkage(d_, best_k)) == 1.0)
        accs = [accuracy_score(y_ref[val_ids], RandomForestClassifier(n_estimators=100, min_samples_leaf=2, random_state=s).fit(X_train, y_ref[train_ids]).predict(X_val)) for s in range(30)]
        print(f"scikit-learn's forest (100 trees, leaf 2), validation accuracy over 30 seeds: {np.mean(accs):.3f} +- {np.std(accs):.3f} (range {min(accs):.3f} to {max(accs):.3f}); ours: {np.mean(res[(100, 2)]):.3f} +- {np.std(res[(100, 2)]):.3f}")
        check("our forest and scikit-learn's forest agree in mean validation accuracy over 30 seeds (within 0.1)", abs(np.mean(accs) - np.mean(res[(100, 2)])) < 0.1)
    except Exception as e: print("stage could not be completed:", type(e).__name__, e)

In [ ]:
HAVE_RD = importlib.util.find_spec("rdkit") is not None
if not HAVE_RD and HAVE_SK:
    try:
        import subprocess, sys; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdkit"]); HAVE_RD = importlib.util.find_spec("rdkit") is not None
    except Exception as e: print("RDKit could not be installed:", type(e).__name__, e)
if HAVE_RD:
    try:
        from rdkit import Chem
        from rdkit.Chem import Descriptors, rdMolDescriptors, Crippen, Lipinski
        rd = pd.DataFrame([{"MW": Descriptors.MolWt(m), "logP": Crippen.MolLogP(m), "TPSA": rdMolDescriptors.CalcTPSA(m), "HBD": Lipinski.NumHDonors(m), "HBA": Lipinski.NumHAcceptors(m), "rings": rdMolDescriptors.CalcNumRings(m), "fraction_Csp3": rdMolDescriptors.CalcFractionCSP3(m), "oxygen_atoms": sum(a.GetAtomicNum() == 8 for a in m.GetAtoms())} for m in (Chem.MolFromSmiles(s) for s in data.smiles)])
        diff = (rd - features).abs().max(); print("largest difference per descriptor, RDKit against ours:\n" + diff.round(4).to_string()); print("molecules with a logP difference above 1e-3:", [data.name[i] for i in np.where((rd.logP - features.logP).abs() > 1e-3)[0]])
        check("all eight descriptors of all 40 molecules equal RDKit's (the three acids whose logP is estimated by group contributions included)", bool((diff < 1e-3).all()))
    except Exception as e: print("RDKit stage could not be completed:", type(e).__name__, e)
else: print("RDKit is not available: the RDKit check was skipped.")
if checks:
    ok = [lab for lab, p in checks if p]; bad = [lab for lab, p in checks if not p]; print(f"\nPart II: {len(ok)} of {len(checks)} checks passed")
    for lab in bad: print("  not passed:", lab)

### What to look for
* **Our descriptors against RDKit's.** If all eight descriptors of all 40 molecules agree, the molecular graph, the hydrogens and the atom-type rules of Part I are the same as RDKit's; the three acids with an estimated $\log P$ (formic, glycolic, lactic) are the ones to read in the printed table.
* **The notebook's forest numbers.** They are reproduced when scikit-learn's forest runs on this descriptor table (the numbers depend on the scikit-learn version).
* **The seed.** Compare the spread of the forest's validation accuracy over seeds with the difference between the settings of the notebook's table.